# BT4012 Group 05 — Colab runner
Runs the heavy steps (5-seed training sweeps, self-training, adversarial injection) on a Colab GPU/CPU.
1. Runtime → Change runtime type → GPU (optional; everything also runs on CPU).
2. Fill in the repo URL. Colab ships Python 3.13, newer than the project pin (`<3.13`), so the package is
   not pip-installed; the scripts add `src/` to `sys.path` themselves and only the dependencies are installed.
3. LLM scoring is cached in `data/llm/*.parquet` (committed); the API key is only needed to extend the cache (never commit it).

In [ ]:
REPO = "https://github.com/ykwong2002/bt4012-adaptive-fraud-detection.git"
!git clone -q $REPO repo
%cd repo
# Colab already has numpy, pandas, scipy, scikit-learn, networkx, torch, matplotlib, tqdm, pyarrow
!pip install -q torch_geometric catboost sentence-transformers anthropic pyyaml python-dotenv

In [ ]:
# YelpZip raw release: place metadata + reviewContent under data/raw/yelpzip_raw/ (see data/README.md),
# e.g. by mounting Drive and copying, or uploading via the Files pane.
# from google.colab import drive; drive.mount('/content/drive')
# !mkdir -p data/raw/yelpzip_raw && cp /content/drive/MyDrive/yelpzip/{metadata,reviewContent} data/raw/yelpzip_raw/
!python scripts/01_download.py   # public YelpChi.mat for the graph-only track (optional)

In [ ]:
!python scripts/02_build_graph.py --source raw
!python scripts/03_eda.py --source raw

In [ ]:
# LLM scoring is cached in data/llm/*.parquet (committed). Only run this if the cache is incomplete.
# import os; os.environ['ANTHROPIC_API_KEY'] = '...'
# !python scripts/04_llm_score.py --source raw --dry-run

In [ ]:
!python scripts/05_distil_edges.py --source raw
!python scripts/05c_edge_diagnostics.py --source raw
!python scripts/06_train.py --source raw
!python scripts/06_train.py --source raw --weights data/processed/edge_scores_raw.npz --mode soft
!python scripts/06_train.py --source raw --weights data/processed/edge_scores_raw.npz --mode softrank
!python scripts/06_train.py --source raw --weights data/processed/edge_scores_raw.npz --mode hard --tau 0.5

In [ ]:
!python scripts/07_drift.py --source raw
!python scripts/07_drift.py --source raw --same-window
!python scripts/08_selftrain.py --source raw
!python scripts/08_selftrain.py --source raw --weights data/processed/edge_scores_raw.npz
!python scripts/09_adversarial.py --source raw --recover
!python scripts/12_stats.py --tag raw
!python scripts/10_figures.py --tag raw

In [ ]:
# bring results back: zip outputs/ and download, or commit outputs/tables + outputs/figures on a branch
!zip -qr outputs.zip outputs/tables outputs/figures outputs/runs && echo done